# Финальный depth=6 и сезонный submission

Ноутбук обучает лучшую проверенную базовую конфигурацию CatBoost на всех восьми размеченных срезах, строит прогноз для якоря `2026-02-13` и применяет к нему сезонную residual-коррекцию. Сохраняются два CSV, чтобы отдельно измерить эффект корректора на leaderboard.

## 0. Режим запуска

`TRAIN_FINAL_BASE=True` переобучает финальный CatBoost. `BUILD_SUBMISSIONS=True` пересобирает оба CSV. После первого полного запуска флаги выключаются.

In [1]:
TRAIN_FINAL_BASE = False
BUILD_SUBMISSIONS = False

FINAL_ITERATIONS = 822
CORRECTION_LIMIT = 0.20

## 1. Импорты и пути

In [2]:
from __future__ import annotations

import gc
import json
from pathlib import Path
import sys

from catboost import CatBoostRegressor
import numpy as np
import pandas as pd
import polars as pl

project_root = Path.cwd().resolve()
if not (project_root / 'src').is_dir():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import (
    CATBOOST_CORRECTION_DIR,
    CATBOOST_SEASONALITY_DIR,
    CATBOOST_V2_SNAPSHOT_DIR,
    DEPTH6_SUBMISSION_PATH,
    FINAL_SUBMISSION_ARTIFACT_DIR,
    SEASONAL_SUBMISSION_PATH,
    ensure_output_dirs,
)
from src.correction import apply_log_correction
from src.evaluation import save_json
from src.features import load_snapshots
from src.models import make_final_model, predict_gmv
from src.validation import feature_columns

ensure_output_dirs()
BASE_MODEL_PATH = FINAL_SUBMISSION_ARTIFACT_DIR / 'catboost_depth6_822.cbm'
BASE_PREDICTION_PATH = FINAL_SUBMISSION_ARTIFACT_DIR / 'depth6_822_predictions.parquet'
SUMMARY_PATH = FINAL_SUBMISSION_ARTIFACT_DIR / 'summary.json'

## 2. Готовые матрицы признаков

Базовая модель использует те же 216 `enhanced_v2` признаков. В обучение входят все 8 исторических якорей, то есть 2 миллиона строк. Финальный срез не содержит target.

In [3]:
snapshots = load_snapshots(CATBOOST_V2_SNAPSHOT_DIR, kind='train')
historical_anchors = sorted(snapshots)
test_path = CATBOOST_V2_SNAPSHOT_DIR / 'test_2026-02-13.parquet'
test_snapshot = pl.read_parquet(test_path)
features = feature_columns(snapshots[historical_anchors[0]])

assert len(snapshots) == 8
assert len(features) == 216
assert feature_columns(test_snapshot) == features
assert test_snapshot.height == 250_000
for anchor, snapshot in snapshots.items():
    assert feature_columns(snapshot) == features, f'Отличается схема {anchor}'

print('Якори:', historical_anchors)
print(f'Обучающих строк: {sum(s.height for s in snapshots.values()):,}')
print(f'Признаков: {len(features)}')

Якори: [datetime.date(2025, 7, 2), datetime.date(2025, 7, 30), datetime.date(2025, 8, 27), datetime.date(2025, 9, 24), datetime.date(2025, 10, 22), datetime.date(2025, 11, 19), datetime.date(2025, 12, 17), datetime.date(2026, 1, 14)]
Обучающих строк: 2,000,000
Признаков: 216


## 3. Финальный CatBoost

`depth=6` выбран по четырёхфолдовой временной валидации. На ближайшем к финалу январском holdout лучшей была итерация 722. Для финальной модели используется 822 дерева: тот же консервативный запас +100, который уже заложен в проекте.

In [4]:
if TRAIN_FINAL_BASE:
    train = pl.concat([snapshots[a] for a in historical_anchors], how='vertical_relaxed')
    x_train = train.select(features).to_pandas()
    y_train = np.log1p(train['target'].to_numpy())
    x_test = test_snapshot.select(features).to_pandas()

    base_model = make_final_model(FINAL_ITERATIONS, depth=6)
    base_model.fit(x_train, y_train)
    base_prediction = predict_gmv(base_model, x_test)
    base_model.save_model(str(BASE_MODEL_PATH))
    pd.DataFrame({
        'user_id': test_snapshot['user_id'].to_numpy(),
        'prediction': base_prediction,
    }).to_parquet(BASE_PREDICTION_PATH, index=False)
    del train, x_train, y_train, x_test, base_model
    gc.collect()
else:
    if not BASE_PREDICTION_PATH.exists() or not BASE_MODEL_PATH.exists():
        raise FileNotFoundError('Нет финальной модели. Установите TRAIN_FINAL_BASE=True.')
    base_prediction = pd.read_parquet(BASE_PREDICTION_PATH)['prediction'].to_numpy()

assert len(base_prediction) == test_snapshot.height
assert np.isfinite(base_prediction).all()
assert (base_prediction >= 0).all()
pd.Series(base_prediction, name='base_prediction').describe(percentiles=[0.5, 0.9, 0.99, 0.999])

0:	learn: 2.2905541	total: 1.55s	remaining: 21m 12s


200:	learn: 1.7112873	total: 4m 10s	remaining: 12m 54s


400:	learn: 1.7082460	total: 9m 7s	remaining: 9m 35s


600:	learn: 1.7063774	total: 13m 1s	remaining: 4m 47s


800:	learn: 1.7048884	total: 16m 22s	remaining: 25.8s


821:	learn: 1.7047425	total: 16m 50s	remaining: 0us


count    250000.000000
mean         38.489327
std         101.754226
min           0.000000
50%           7.299005
90%          97.303230
99%         457.493220
99.9%      1225.671825
max        3741.405103
Name: base_prediction, dtype: float64

## 4. Сезонная коррекция

Готовый корректор получает базовый `log1p`-прогноз, 216 текущих признаков и 59 year-over-year признаков. Он предсказывает поправку в `log1p`; поправка ограничивается диапазоном `[-0.2, +0.2]`.

In [5]:
seasonal_path = CATBOOST_SEASONALITY_DIR / 'features_2026-02-13.parquet'
seasonal_snapshot = pl.read_parquet(seasonal_path)
seasonal_features = [
    c for c in seasonal_snapshot.columns if c not in {'user_id', 'anchor_date'}
]
assert len(seasonal_features) == 59

correction_frame = pd.DataFrame({
    'user_id': test_snapshot['user_id'].to_numpy(),
    'base_log_prediction': np.log1p(base_prediction),
})
correction_frame = correction_frame.merge(
    test_snapshot.select(['user_id', *features]).to_pandas(),
    on='user_id', how='left', validate='one_to_one',
).merge(
    seasonal_snapshot.select(['user_id', *seasonal_features]).to_pandas(),
    on='user_id', how='left', validate='one_to_one',
)
correction_columns = ['base_log_prediction', *features, *seasonal_features]

correction_model = CatBoostRegressor()
correction_model.load_model(str(CATBOOST_CORRECTION_DIR / 'seasonal_correction.cbm'))
assert correction_model.feature_names_ == correction_columns
raw_correction = correction_model.predict(correction_frame[correction_columns])
seasonal_prediction = apply_log_correction(
    base_prediction, raw_correction, correction_limit=CORRECTION_LIMIT
)
correction_diagnostics = pd.Series(raw_correction, name='raw_correction').describe(
    percentiles=[0.01, 0.1, 0.5, 0.9, 0.99]
)
display(correction_diagnostics)
print('Доля на нижнем ограничении:', float((raw_correction <= -CORRECTION_LIMIT).mean()))
print('Доля на верхнем ограничении:', float((raw_correction >= CORRECTION_LIMIT).mean()))

count    250000.000000
mean         -0.248451
std           0.156221
min          -0.904124
1%           -0.567464
10%          -0.451247
50%          -0.243968
90%          -0.060043
99%           0.174868
max           0.636577
Name: raw_correction, dtype: float64

Доля на нижнем ограничении: 0.605896
Доля на верхнем ограничении: 0.007248


## 5. Сборка двух submission

Порядок `user_id` берётся из финального среза. Каждый CSV должен содержать ровно 250 000 уникальных пользователей и неотрицательный `predict`.

In [6]:
base_submission = pd.DataFrame({
    'user_id': test_snapshot['user_id'].to_numpy(),
    'predict': np.clip(base_prediction, 0, None),
})
seasonal_submission = pd.DataFrame({
    'user_id': test_snapshot['user_id'].to_numpy(),
    'predict': np.clip(seasonal_prediction, 0, None),
})

for label, submission in {
    'base': base_submission,
    'seasonal': seasonal_submission,
}.items():
    assert list(submission.columns) == ['user_id', 'predict']
    assert submission.shape == (250_000, 2)
    assert submission['user_id'].is_unique
    assert np.isfinite(submission['predict']).all()
    assert (submission['predict'] >= 0).all()

if BUILD_SUBMISSIONS:
    base_submission.to_csv(DEPTH6_SUBMISSION_PATH, index=False, float_format='%.8f')
    seasonal_submission.to_csv(SEASONAL_SUBMISSION_PATH, index=False, float_format='%.8f')
else:
    if not DEPTH6_SUBMISSION_PATH.exists() or not SEASONAL_SUBMISSION_PATH.exists():
        raise FileNotFoundError('Нет готовых CSV. Установите BUILD_SUBMISSIONS=True.')

print('Базовый:', DEPTH6_SUBMISSION_PATH)
print('Сезонный:', SEASONAL_SUBMISSION_PATH)

Базовый: /Users/danasokol/Desktop/ML - соревы/OZON_GMV/submissions/submission_depth6_822.csv
Сезонный: /Users/danasokol/Desktop/ML - соревы/OZON_GMV/submissions/submission_depth6_822_seasonal_correction.csv


## 6. Диагностика и сохранение конфигурации

In [7]:
prediction_comparison = pd.DataFrame({
    'base': base_submission['predict'].describe(percentiles=[0.5, 0.9, 0.99]),
    'seasonal': seasonal_submission['predict'].describe(percentiles=[0.5, 0.9, 0.99]),
})
display(prediction_comparison)

summary = {
    'final_anchor': '2026-02-13',
    'forecast_period': ['2026-02-14', '2026-03-15'],
    'base_depth': 6,
    'base_iterations': FINAL_ITERATIONS,
    'n_train_anchors': len(snapshots),
    'n_train_rows': int(sum(s.height for s in snapshots.values())),
    'n_base_features': len(features),
    'n_seasonal_features': len(seasonal_features),
    'correction_limit_log1p': CORRECTION_LIMIT,
    'mean_base_prediction': float(base_submission['predict'].mean()),
    'mean_seasonal_prediction': float(seasonal_submission['predict'].mean()),
    'median_base_prediction': float(base_submission['predict'].median()),
    'median_seasonal_prediction': float(seasonal_submission['predict'].median()),
    'lower_clip_share': float((raw_correction <= -CORRECTION_LIMIT).mean()),
    'upper_clip_share': float((raw_correction >= CORRECTION_LIMIT).mean()),
    'base_submission': str(DEPTH6_SUBMISSION_PATH),
    'seasonal_submission': str(SEASONAL_SUBMISSION_PATH),
}
save_json(summary, SUMMARY_PATH)
display(pd.DataFrame([summary]))

,base,seasonal
count,250000.000000,250000.000000
mean,38.489327,32.866105
std,101.754226,89.643489
min,0.000000,0.000000
50%,7.299005,6.010174
90%,97.303230,82.610133
99%,457.493220,396.421499
max,3741.405103,3455.499641


,final_anchor,forecast_period,base_depth,base_iterations,n_train_anchors,n_train_rows,n_base_features,n_seasonal_features,correction_limit_log1p,mean_base_prediction,mean_seasonal_prediction,median_base_prediction,median_seasonal_prediction,lower_clip_share,upper_clip_share,base_submission,seasonal_submission
0,2026-02-13,"[2026-02-14, 2026-03-15]",6,822,8,2000000,216,59,0.2,38.489327,32.866105,7.299005,6.010174,0.605896,0.007248,/Users/danasokol/Desktop/ML - соревы/OZON_GMV/...,/Users/danasokol/Desktop/ML - соревы/OZON_GMV/...


## 7. Итог

Финальный CatBoost `depth=6`, `iterations=822` обучен на 2 000 000 строк из восьми временных срезов. Созданы два submission для периода `2026-02-14` — `2026-03-15`.

Базовый depth=6 имеет средний прогноз **38.49** и медиану **7.30**. После сезонной коррекции среднее снижается до **32.87**, а медиана — до **6.01**. У **60.59%** пользователей понижающая поправка упирается в ограничение `-0.2`, у **0.72%** — в верхнее `+0.2`.

Первым логично отправить **базовый depth=6**: его архитектура полностью проверена на четырёх временных holdout. Сезонный CSV — отдельный более рисковый leaderboard-эксперимент, потому что корректор обучен только на одном зимнем якоре.